# Task 2 - Independent Season Evaluation

**Owner:** Kai  
**Status:** locked replay scaffold; holdout scoring starts only after the group freeze receipt exists.

This notebook evaluates the frozen I2 Season bundle. It does not retrain, retune, change the winner, or create a new split. The notebook is intentionally safe to run before the holdout unlock: the default mode reads development evidence and checks the sealed handoff without reading protected labels.

## 1. Independent evaluation contract

The internal holdout is a one-shot evaluation set. The final model choice must be frozen before its labels are read. The official teacher test set is separate and is not used for this scorecard.

The holdout primary comparator is **B0 training-fold majority**. B1 HOG + HSV LinearSVC is a useful secondary comparator only if its holdout predictions were frozen before labels were opened. The development selection story remains I2 versus C2; it is not re-run here.

### 1.1 Set replay mode and project paths

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

EVALUATION_MODE = 'replay'  # change to 'score' only after the group unlock receipt
SEED = 2753
HOLDOUT_ROWS_EXPECTED = 5778
QUARANTINE_ROWS_EXPECTED = 61
LABELS = ('Fall', 'Spring', 'Summer', 'Winter')
SELECTION_PATH = ROOT / 'results/evidence/task2/selection_freeze.json'
HANDOFF_PATH = ROOT / 'results/evidence/task2/final_handoff/manifest.json'
MODEL_MANIFEST_PATH = ROOT / 'models/task2_season.manifest.json'
MODEL_PATH = ROOT / 'models/task2_season.pt'
print({'root': ROOT.as_posix(), 'mode': EVALUATION_MODE, 'labels': LABELS})

**Interpretation.** `replay` is the safe default. It can verify frozen development evidence, but it cannot claim a holdout result. Switching to `score` is a controlled action that requires the group receipt and the prediction receipt.

## 2. Frozen I2 bundle and provenance

### 2.1 Load and verify the pre-holdout freeze

In [ ]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

selection = read_json(SELECTION_PATH)
handoff = read_json(HANDOFF_PATH)
model_manifest = read_json(MODEL_MANIFEST_PATH)
assert selection['status'] == 'frozen'
assert selection['selected_model']['candidate'] == 'I2'
assert selection['selected_model']['scratch'] is True
assert selection['selected_model']['weights'] is None
assert selection['selected_model']['inference_inputs'] == ['image']
assert handoff['selected_candidate'] == 'I2'
assert handoff['holdout_opened'] is False
assert handoff['evaluation_claim_allowed'] is False
assert model_manifest['holdout_opened'] is False
print({
    'candidate': selection['selected_model']['candidate'],
    'experiment_id': selection['selected_model']['experiment_id'],
    'model_family': selection['selected_model']['model_family'],
    'parameters': selection['selected_model']['parameter_count'],
    'bundle_sha256': sha256(MODEL_PATH),
    'manifest_sha256': sha256(MODEL_MANIFEST_PATH),
    'group_freeze_verified': handoff['group_freeze_verified'],
    'holdout_opened': handoff['holdout_opened'],
})

**Interpretation.** The frozen bundle is a scratch SmallCNN. ArticleType was used only as a masked auxiliary training target; inference still consumes the image only. The handoff is correctly not yet an evaluation claim because the group freeze has not been verified.

### 2.2 Freeze metric, calibration, and baseline policy

In [ ]:
primary_metric = selection['primary_development_evidence']['metric']
temperature = selection['calibration']['temperature']
b0_role = 'primary independent sanity comparator'
b1_role = 'secondary only when pre-frozen holdout predictions exist'
assert primary_metric == 'pooled_five_fold_oof_macro_f1'
assert math.isfinite(float(temperature)) and float(temperature) > 0
print({
    'primary_metric': primary_metric,
    'temperature': temperature,
    'B0': b0_role,
    'B1': b1_role,
    'review_threshold': selection['calibration']['app_review_threshold'],
})

**Interpretation.** B0 is the correct primary holdout baseline because it is deterministic and asks the first practical question: does the image model beat the class prior? B1 is stronger but is not allowed to appear as a newly trained post-holdout comparison.

## 3. Development evidence before holdout

### 3.1 Reproduce the incremental selection story

In [ ]:
import pandas as pd

scorecard = pd.read_csv(ROOT / 'results/evidence/task2/ultimate_judgement/scorecard.csv')
development_baselines = pd.DataFrame([
    {'model': 'B0 majority', 'role': 'prior baseline', 'macro_f1': pd.read_json(ROOT / 'results/evidence/task2/b0_majority/pooled_metrics.json', typ='series')['macro_f1']},
    {'model': 'B1 HOG + HSV SVM', 'role': 'classical image baseline', 'macro_f1': pd.read_json(ROOT / 'results/evidence/task2/b1_hog_hsv_svm/pooled_metrics.json', typ='series')['macro_f1']},
    {'model': 'C2 scratch ResNet18', 'role': 'development reference', 'macro_f1': scorecard.loc[scorecard['candidate'].eq('C2'), 'primary_macro_f1'].iloc[0]},
    {'model': 'I2 scratch SmallCNN + ArticleType auxiliary loss', 'role': 'frozen winner', 'macro_f1': scorecard.loc[scorecard['candidate'].eq('I2'), 'primary_macro_f1'].iloc[0]},
])
development_baselines

**Interpretation.** The model story is incremental: B0 establishes the class-prior floor, B1 tests hand-crafted visual features, C2 tests a deeper scratch family, and I2 adds a training-only related target to improve the difficult Season decision. Holdout evaluation must not restart this search.

### 3.2 Display the development learning and selection evidence

In [ ]:
import matplotlib.pyplot as plt

fig, axis = plt.subplots(figsize=(8, 4.5))
axis.bar(development_baselines['model'], development_baselines['macro_f1'], color=['#9aa0a6', '#5f8dd3', '#8a6bbf', '#d65f5f'])
axis.set_ylabel('Development macro-F1')
axis.set_title('Incremental Season model selection before holdout')
axis.tick_params(axis='x', rotation=18)
axis.set_ylim(0, 0.85)
fig.tight_layout()
figure_path = ROOT / 'results/figures/task2/final_evaluation/development_selection_story.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=160)
plt.show()
print(figure_path.relative_to(ROOT).as_posix())

**Interpretation.** This figure is a pre-holdout explanation, not a holdout result. It supports Assessment 3's critical-analysis section by showing why I2 was selected and which alternatives were rejected.

## 4. Holdout prediction receipt and one-shot unlock

### 4.1 Preflight the sealed holdout without labels

In [ ]:
from fashion.data.dataset import load_splits

sealed = load_splits()
holdout = sealed.loc[sealed['partition'].eq('holdout')].copy()
quarantine = sealed.loc[sealed['partition'].eq('quarantine')].copy()
assert len(holdout) == HOLDOUT_ROWS_EXPECTED
assert len(quarantine) == QUARANTINE_ROWS_EXPECTED
assert holdout['id'].is_unique
assert quarantine['id'].is_unique
assert all(str(value).strip() == '' for value in holdout['season'].unique())
assert all(str(value).strip() == '' for value in quarantine['season'].unique())
print({
    'holdout_rows': len(holdout),
    'quarantine_rows_excluded': len(quarantine),
    'holdout_labels_visible': bool(holdout['season'].astype(str).str.strip().ne('').any()),
    'unique_holdout_ids': int(holdout['id'].nunique()),
})

**Interpretation.** The canonical loader exposes the holdout membership and image paths but redacts protected targets. This cell proves that the prediction phase can cover all 5,778 holdout rows without using Season labels.

### 4.2 Score only after the group receipt exists

In [ ]:
def require_group_unlock(handoff_payload: dict) -> None:
    if not handoff_payload.get('group_freeze_verified', False):
        raise RuntimeError('group freeze is not verified; do not open protected labels')
    if not handoff_payload.get('notebook_06_unlocked', False):
        raise RuntimeError('Notebook 06 is not unlocked by the group receipt')

if EVALUATION_MODE == 'score':
    require_group_unlock(handoff)
    raise NotImplementedError('Controlled scoring is enabled only after the shared prediction receipt is committed')
else:
    print('LOCKED: replay mode does not open protected labels or create holdout scores.')

**Interpretation.** A controlled scoring implementation will be added only after every task owner provides a label-free prediction receipt. This guard prevents a personal notebook from creating a second, uncontrolled holdout unlock.

## 5. Planned holdout scorecard

### 5.1 Frozen metric table

In [ ]:
planned_metrics = [
    'macro_f1', 'balanced_accuracy', 'accuracy', 'weighted_f1',
    'macro_precision', 'macro_recall', 'nll', 'brier', 'ece'
]
planned_scorecard = pd.DataFrame({
    'model': ['B0 majority', 'I2 frozen SmallCNN'],
    'role': ['primary holdout sanity baseline', 'final selected model'],
    'macro_f1': [pd.NA, pd.NA],
    'balanced_accuracy': [pd.NA, pd.NA],
    'status': ['awaiting one-shot scoring', 'awaiting one-shot scoring'],
})
planned_scorecard

**Interpretation.** The table is deliberately empty until the controlled scoring receipt is available. It prevents development metrics from being mislabeled as independent evidence. B1 can be added as a third row only when its prediction artifact was frozen before unlock.

## 6. Analysis contract for the final result

### 6.1 Required error, shift, and practical-viability evidence

In [ ]:
analysis_contract = {
    'per_class': list(LABELS),
    'slices': ['Spring', 'ArticleType aligned/conflict', 'acquisition year', 'development-fitted file-size quartile', 'family size', 'grayscale/RGB'],
    'robustness': ['clean', 'jpeg_quality_85', 'brightness_0.85', 'brightness_1.15', 'gaussian_blur_radius_1'],
    'uncertainty': '10,000 product-family bootstrap draws, seed 2753, 95% percentile intervals',
    'calibration': ['NLL', 'Brier', 'ECE-15', 'reliability diagram', 'risk-coverage diagnostic'],
    'cost': ['CPU latency', 'GPU latency when available', 'parameter count', 'memory', 'prediction failures'],
    'decision_boundary': 'catalogue decision support with human review for uncertain or shifted images',
}
pd.Series(analysis_contract, name='frozen_contract')

**Interpretation.** The final judgement must go beyond one accuracy number. It must show where I2 works, where it fails, how confident it is, how much shift it tolerates, and whether its cost fits a real catalogue workflow.

## 7. Assessment 3 connection

### 7.1 Literature argument for the presentation

In [ ]:
literature = pd.DataFrame([
    {'source': 'Seo et al. (2025)', 'lesson': 'same broad fashion source, but different target, resolution, pretrained and multimodal inputs', 'direct_score_comparison': False},
    {'source': 'Kolisnik et al. (2021)', 'lesson': 'hierarchical fashion labels show useful structure, but not a direct Season benchmark', 'direct_score_comparison': False},
    {'source': 'Ferreira et al. (2018)', 'lesson': 'structured multi-task fashion prediction motivates related auxiliary targets', 'direct_score_comparison': False},
    {'source': 'Multimodal Sequential Fashion Attribute Prediction (2019)', 'lesson': 'Season is contextual and dataset-dependent', 'direct_score_comparison': False},
    {'source': 'Guo et al. (2017); Ovadia et al. (2019)', 'lesson': 'confidence must be evaluated separately and may degrade under shift', 'direct_score_comparison': False},
    {'source': 'Koh et al. (2021)', 'lesson': 'real-world shift requires independent and slice-aware evaluation', 'direct_score_comparison': False},
], columns=['source', 'lesson', 'direct_score_comparison'])
literature

**Interpretation.** The papers support method choices and evaluation boundaries; they are not interchangeable benchmarks. Assessment 3 should explain what each paper contributes, what it cannot prove for this dataset, and why I2 is still evaluated on the project's own frozen split.

### 7.2 Extension to present and demonstrate

In [ ]:
extension_plan = pd.DataFrame([
    ['Problem', 'Low-light and shifted catalogue images cause overconfident Season errors, especially Spring.'],
    ['Approach', 'Add an image-quality/OOD gate and route high-risk predictions to a human reviewer.'],
    ['Evidence', 'Brightness stress, calibration, confidence, risk-coverage, and deterministic error examples.'],
    ['Success', 'Higher macro-F1 and Spring recall on a future shifted set, lower false-confidence rate, and measurable review cost.'],
], columns=['part', 'proposal'])
extension_plan

**Interpretation.** The extension is a concrete application, not a vague request to use a larger model. It has a defined failure scenario, a proposed human-in-the-loop method, and measurable success criteria.

## 8. Handoff to shared Notebook 06

In [ ]:
handoff_checklist = pd.DataFrame([
    ['Frozen I2 bundle', 'pass' if handoff['selected_candidate'] == 'I2' else 'fail'],
    ['Scratch weights', 'pass' if model_manifest['weights'] is None else 'fail'],
    ['Image-only inference', 'pass' if model_manifest['loader_audit']['auxiliary_target'] == 'articleType' else 'review'],
    ['Holdout labels opened', 'not yet' if not handoff['holdout_opened'] else 'record receipt'],
    ['Independent claim', 'blocked until one-shot scoring'],
    ['Shared Notebook 06 input', 'this notebook plus immutable evaluation manifest'],
], columns=['gate', 'status'])
handoff_checklist

**Final interpretation.** This owner notebook is ready to receive the controlled holdout evidence. Until then, the correct conclusion is only that I2 is the frozen development winner and that independent evaluation is still pending.